Pipeline — Mercedes-Benz Test Bench Prediction

## 1. Load raw data

Import the libraries we'll need and load the raw training set.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the raw training data
df_train = pd.read_csv("../data/raw/train.csv")

print(f"Shape: {df_train.shape}")
df_train.head()

Shape: (4209, 378)


,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0


## 2. Split target from features

`y` is what we want to predict. `X` is everything else, except `ID` (just a row identifier with no predictive value).

In [2]:
y = df_train['y']
X = df_train.drop(['ID', 'y'], axis=1)

print(f"y shape: {y.shape}")
print(f"X shape: {X.shape}")

y shape: (4209,)
X shape: (4209, 376)


## 3. Train/validation split

Hold out 20% of the data as validation, so we can honestly measure how well the model generalizes to data it never saw during training. Using `random_state=42` so this split is reproducible and matches the rest of the team's work.

In [3]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")

X_train shape: (3367, 376)
X_val shape:   (842, 376)


## 4. Identify and remove the outlier



In [4]:
# Look at the distribution of y to spot anything unusual
print(y.describe())

count    4209.000000
mean      100.669318
std        12.679381
min        72.110000
25%        90.820000
50%        99.150000
75%       109.010000
max       265.320000
Name: y, dtype: float64


The max (265.32) is far above the 75th percentile (109.01) — roughly 13 standard deviations above the mean. That's a strong signal of an outlier. Let's find exactly which row it is.

In [5]:
# Find the row(s) with unusually high y values
outliers = df_train[df_train['y'] > 200]
outliers[['ID', 'y']]

,ID,y
883,1770,265.32


Confirmed: `ID=1770` has `y=265.32` (row index 883 is just its position in the table, not the actual identifier). This row falls in `X_train` under our `random_state=42` split, so we drop it from training only — `X_val` stays untouched.

In [6]:
outlier_id = 1770
outlier_mask = df_train.loc[X_train.index, 'ID'] == outlier_id

X_train = X_train[~outlier_mask]
y_train = y_train[~outlier_mask]

print(f"X_train shape after removing outlier: {X_train.shape}")

X_train shape after removing outlier: (3366, 376)


## 5. Identify constant and duplicate columns

A constant column has the same value in every row — no information for the model. A duplicate column has exactly the same values as another column — redundant, so we only need to keep one.

We calculate this on `X_train` only (after removing the outlier), not on the full dataset, to avoid leaking any information from validation into this decision — the same "fit on train only" principle used for encoding.

In [7]:
# Separate categorical from binary columns
cat_cols = X_train.select_dtypes(include='str').columns.tolist()
binary_cols = [col for col in X_train.columns if col not in cat_cols]

print(f"Categorical columns: {len(cat_cols)}")
print(f"Binary columns: {len(binary_cols)}")

# Find constant columns within the binary block
constant_cols = [col for col in binary_cols if X_train[col].nunique() == 1]
print(f"\nConstant columns: {len(constant_cols)}")

# Find duplicate columns within the non-constant binary block
non_constant_binary = [col for col in binary_cols if col not in constant_cols]
duplicate_groups = X_train[non_constant_binary].T.groupby(
    X_train[non_constant_binary].T.apply(tuple, axis=1)
).groups
duplicate_sets = [list(v) for v in duplicate_groups.values() if len(v) > 1]
duplicate_cols_to_drop = [col for group in duplicate_sets for col in group[1:]]

print(f"Duplicate columns to drop: {len(duplicate_cols_to_drop)}")

cols_to_drop = constant_cols + duplicate_cols_to_drop
print(f"\nTotal columns to drop: {len(cols_to_drop)}")

Categorical columns: 8
Binary columns: 368

Constant columns: 13
Duplicate columns to drop: 48

Total columns to drop: 61


## 6. Drop the identified columns from all three sets

We decided which columns to drop using `X_train` only. Now we apply that same list to `X_train`, `X_val`, and `X_test` — same principle as before: decide on train, apply everywhere.

Also loading `X_test` here for the first time, since we'll need it for this step and for generating predictions later.

In [8]:
# Load the test set (needed to apply the same cleaning)
df_test = pd.read_csv("../data/raw/test.csv")
test_ID = df_test['ID']
X_test = df_test.drop(['ID'], axis=1)

# Drop the identified columns from all three sets
X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (3366, 315)
X_val shape:   (842, 315)
X_test shape:  (4209, 315)


## 7. Encode categorical columns

The 8 categorical columns are still text — models only understand numbers. We use Ordinal Encoding (not One-Hot) because some columns have up to 47 categories (One-Hot would create too many new columns), and tree-based models like XGBoost don't assume any real order in the numbers, so there's no downside.

The encoder is fit on `X_train` only (avoiding leakage), and `handle_unknown='use_encoded_value', unknown_value=-1` safely handles categories that appear in `X_val`/`X_test` but were never seen in `X_train`.

In [9]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Recompute cat_cols/num_cols on the cleaned X_train
cat_cols = X_train.select_dtypes(include='str').columns.tolist()
num_cols = X_train.select_dtypes(exclude='str').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

preprocessor.set_output(transform='pandas')
preprocessor.fit(X_train)

X_train_encoded = preprocessor.transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print(f"X_train_encoded shape: {X_train_encoded.shape}")

X_train_encoded shape: (3366, 315)


## 8. Baseline model — predict the mean


In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

# The simplest possible model: always predict the mean of y
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train_encoded, y_train)

y_val_pred_baseline = baseline_model.predict(X_val_encoded)
r2_baseline = r2_score(y_val, y_val_pred_baseline)

print(f"Baseline R2 (predict the mean): {r2_baseline:.4f}")

Baseline R2 (predict the mean): -0.0000


## 9. Linear Regression


In [11]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)

y_val_pred_linear = linear_model.predict(X_val_encoded)
r2_linear = r2_score(y_val, y_val_pred_linear)

print(f"Linear Regression R2: {r2_linear:.4f}")
print(f"Improvement over baseline: {r2_linear - r2_baseline:.4f}")

Linear Regression R2: 0.5391
Improvement over baseline: 0.5391



R² = 0.5391 — a big jump from the 0.00 baseline, and notably **higher** than XGBoost's untuned R² (0.4493). This matches what the team observed with Ridge in earlier experiments. Likely explanation: with >90% binary columns, a linear model captures simple additive patterns well, while untuned XGBoost may not yet be using its full potential — this is exactly why hyperparameter tuning (GridSearchCV) is the next step.

## 10. Decision Tree

Next step up in complexity from Linear Regression: instead of a straight line, the model asks a series of yes/no questions about the features to reach a prediction. More flexible, but prone to overfitting if not limited — a known risk with single decision trees.

In [12]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train_encoded, y_train)

y_val_pred_tree = tree_model.predict(X_val_encoded)
r2_tree = r2_score(y_val, y_val_pred_tree)

print(f"Decision Tree R2: {r2_tree:.4f}")
print(f"Improvement over Linear Regression: {r2_tree - r2_linear:.4f}")

Decision Tree R2: 0.0599
Improvement over Linear Regression: -0.4792


### 10b. Decision Tree with limited depth

The unlimited tree scored worse than Linear Regression (0.0599 vs 0.5391) — a clear sign of overfitting: with no depth limit, the tree keeps splitting until it nearly memorizes the training data, capturing noise instead of the real pattern. This is the Bias-Variance Tradeoff in action.

Limiting `max_depth` forces the tree to stop earlier, keeping it simpler and (hopefully) more generalizable.

In [13]:
tree_model_limited = DecisionTreeRegressor(max_depth=5, random_state=42)
tree_model_limited.fit(X_train_encoded, y_train)

y_val_pred_tree_limited = tree_model_limited.predict(X_val_encoded)
r2_tree_limited = r2_score(y_val, y_val_pred_tree_limited)

print(f"Decision Tree (max_depth=5) R2: {r2_tree_limited:.4f}")

Decision Tree (max_depth=5) R2: 0.5633


Limiting depth to 5 improved R² from 0.0599 to 0.5633 — now slightly better than Linear Regression. This directly demonstrates the Bias-Variance Tradeoff: an unconstrained model isn't automatically better, and hyperparameter tuning (limiting complexity) matters more than model choice alone.

In [14]:
# Check the ORIGINAL unlimited tree on both train and validation
y_train_pred_tree = tree_model.predict(X_train_encoded)
r2_train_tree = r2_score(y_train, y_train_pred_tree)

print(f"Decision Tree (no limit) - Train R2:      {r2_train_tree:.4f}")
print(f"Decision Tree (no limit) - Validation R2: {r2_tree:.4f}")

Decision Tree (no limit) - Train R2:      0.9804
Decision Tree (no limit) - Validation R2: 0.0599


### Verifying overfitting properly

A low validation R² alone doesn't prove overfitting — it could just mean the model isn't a good fit. The real test is comparing train vs. validation R²:
- Train R²: 0.9804 (near-perfect — the tree memorized the training data)
- Validation R²: 0.0599 (fails on unseen data)

That gap is the actual signature of overfitting — not just a guess based on the validation number alone.

## 11. Random Forest

Next step up from a single Decision Tree: instead of asking one expert (one tree), we ask many experts (many trees, each trained on a slightly different sample of the data) and average their answers. This combination smooths out individual trees' mistakes and reduces the overfitting risk we just saw with the single unlimited tree.

We check both train and validation R² from the start this time, to immediately see if overfitting is a concern.

In [15]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train_encoded, y_train)

y_train_pred_rf = rf_model.predict(X_train_encoded)
y_val_pred_rf = rf_model.predict(X_val_encoded)

r2_train_rf = r2_score(y_train, y_train_pred_rf)
r2_rf = r2_score(y_val, y_val_pred_rf)

print(f"Random Forest - Train R2:      {r2_train_rf:.4f}")
print(f"Random Forest - Validation R2: {r2_rf:.4f}")

Random Forest - Train R2:      0.9206
Random Forest - Validation R2: 0.4848


Random Forest shows less overfitting than the single unlimited tree (train-val gap of 0.44 vs 0.92), confirming that combining many trees smooths out individual overfitting. However, its default validation R² (0.4848) is still worse than both Linear Regression (0.5391) and the depth-limited Decision Tree (0.5633) — another reminder that untuned complex models don't automatically win. This is exactly why hyperparameter tuning matters more than model choice alone.

## 12. Ridge Regression

Linear Regression with regularization: it penalizes overly large coefficients, which helps stability when features are correlated (like the duplicate-derived columns we cleaned earlier). This is the model the team flagged as suspiciously outperforming XGBoost.

In [16]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(random_state=42)
ridge_model.fit(X_train_encoded, y_train)

y_train_pred_ridge = ridge_model.predict(X_train_encoded)
y_val_pred_ridge = ridge_model.predict(X_val_encoded)

r2_train_ridge = r2_score(y_train, y_train_pred_ridge)
r2_ridge = r2_score(y_val, y_val_pred_ridge)

print(f"Ridge - Train R2:      {r2_train_ridge:.4f}")
print(f"Ridge - Validation R2: {r2_ridge:.4f}")

Ridge - Train R2:      0.6247
Ridge - Validation R2: 0.5564


## 13. XGBoost

Like Random Forest, but instead of training trees independently and averaging them, each new tree is trained specifically to correct the errors of the previous ones — a sequential improvement process ("boosting").

In [20]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train_encoded, y_train)

y_train_pred_xgb = xgb_model.predict(X_train_encoded)
y_val_pred_xgb = xgb_model.predict(X_val_encoded)

r2_train_xgb = r2_score(y_train, y_train_pred_xgb)
r2_xgb = r2_score(y_val, y_val_pred_xgb)

print(f"XGBoost - Train R2:      {r2_train_xgb:.4f}")
print(f"XGBoost - Validation R2: {r2_xgb:.4f}")

XGBoost - Train R2:      0.8923
XGBoost - Validation R2: 0.4371


XGBoost's default validation R² (0.4371) is actually the **worst** among all real models tried — even behind Random Forest. Its train-validation gap (0.8923 vs 0.4371) shows the same overfitting pattern seen in Random Forest and the unlimited Decision Tree.

This doesn't mean XGBoost is a bad choice — it means default settings aren't showing its real potential. Simpler models (Linear Regression, Ridge) perform better "out of the box" here, but XGBoost has more room to improve once properly tuned. This is exactly why hyperparameter tuning via GridSearchCV is the critical next step, not optional polish.

In [21]:
results = pd.DataFrame({
    'Model': ['Baseline (mean)', 'Linear Regression', 'Decision Tree (max_depth=5)', 
              'Random Forest', 'Ridge', 'XGBoost'],
    'Validation R2': [r2_baseline, r2_linear, r2_tree_limited, r2_rf, r2_ridge, r2_xgb]
})

results = results.sort_values('Validation R2', ascending=False).reset_index(drop=True)
results

,Model,Validation R2
0,Decision Tree (max_depth=5),0.563345
1,Ridge,0.556407
2,Linear Regression,0.539106
3,Random Forest,0.484760
4,XGBoost,0.437118
5,Baseline (mean),-0.000007


## Final model comparison

The complete ladder: baseline → simple models → tree-based models, all evaluated on the same cleaned data and validation split.

Key takeaway: with default hyperparameters, simpler models (Decision Tree limited, Ridge, Linear Regression) outperform both ensemble methods (Random Forest, XGBoost). This isn't a sign XGBoost is a bad choice — it's a sign that hyperparameter tuning (next step: GridSearchCV) matters more than model complexity alone.

## Seeing predictions in practice

The R² number is abstract — let's see what it actually means car by car: comparing real test bench times against what the model predicted.

In [22]:
comparison = pd.DataFrame({
    'Actual y (real)': y_val.values[:10],
    'Predicted y (model)': y_val_pred_tree_limited[:10]
})
comparison['Difference'] = comparison['Actual y (real)'] - comparison['Predicted y (model)']

comparison

,Actual y (real),Predicted y (model),Difference
0,97.94,94.001350,3.938650
1,96.41,94.001350,2.408650
2,105.83,112.328533,-6.498533
3,79.09,76.437556,2.652444
4,108.69,112.328533,-3.638533
5,94.60,94.001350,0.598650
6,84.48,94.001350,-9.521350
7,110.24,103.476987,6.763013
8,120.80,103.476987,17.323013
9,122.66,112.328533,10.331467
